# HTS classification of messy engineering BOM text: project demo

**Navya M · CS3807 Deep Learning Laboratory · Shiv Nadar University Chennai**
Code: https://github.com/nzoopxw/hts-messy-bom-qwen

This notebook shows the whole project in four parts:

1. **The data:** real engineer-written part descriptions with US HTS labels.
2. **The results:** baselines vs the fine-tuned model, recomputed from saved predictions.
3. **Live model:** the fine-tuned Qwen2.5-3B, before vs after fine-tuning.
4. **Try it:** type any BOM description.

**Before starting:** Runtime → Change runtime type → **T4 GPU**. Run cell 1 first; it asks for three files:
`test.csv`, `predictions_all.csv`, `qwen_hts_lora_adapter.zip`.
Parts 1–2 need no GPU. Parts 3–4 need the GPU and the adapter.

In [ ]:
# 1. Setup: install libraries and upload the three files
!pip -q install -U "transformers>=4.44" "peft>=0.12" "bitsandbytes>=0.43" accelerate
import os
need = [f for f in ("test.csv", "predictions_all.csv", "qwen_hts_lora_adapter.zip") if not os.path.exists(f)]
if need:
    from google.colab import files
    print("Please upload:", need)
    files.upload()
if os.path.exists("qwen_hts_lora_adapter.zip") and not os.path.isdir("qwen_hts_lora_adapter"):
    !unzip -q qwen_hts_lora_adapter.zip
print({f: os.path.exists(f) for f in ("test.csv", "predictions_all.csv", "qwen_hts_lora_adapter")})

## Part 1: The data

Descriptions were harvested from 111 open-source hardware projects on GitHub and labelled with US HTS codes by looking up each part number on Mouser and Digi-Key. The test set comes from **26 projects never seen in training**.

In [ ]:
import re, numpy as np, pandas as pd
te = pd.read_csv("test.csv", dtype=str).fillna("")
te["chars"] = te.text.str.len()
print(f"Test set: {len(te)} rows from {te.project.nunique()} held-out projects | "
      f"{te.hts.nunique()} distinct codes | median length {int(te.chars.median())} characters")
print(f"Rows written by engineers themselves: {(te.copied_from_distributor.str.lower() != 'true').sum()} of {len(te)}\n")
pd.set_option("display.max_colwidth", 60)
te.sample(12, random_state=1)[["text", "hts", "project"]]

## Part 2: Results

Accuracy at each level of the code hierarchy: 2 digits = chapter, 4 = heading, 6 = international subheading, 8 = US tariff line (sets the duty), 10 = full US code.

In [ ]:
from scipy.stats import binomtest
p = pd.read_csv("predictions_all.csv", dtype=str).fillna("")
METHODS = {"tfidf_knn": "TF-IDF nearest neighbour", "qwen_zero_shot": "Qwen2.5-3B, no fine-tuning",
           "qwen_finetuned_no_aug": "Qwen2.5-3B fine-tuned, no augmentation",
           "qwen_finetuned": "Qwen2.5-3B fine-tuned (final)"}
ok = lambda a, b, k: len(a) >= k and a[:k] == b[:k]
def lead(a, b):
    n = 0
    for x, y in zip(a, b):
        if x != y: break
        n += 1
    return n
rows = []
for col, name in METHODS.items():
    r = {"method": name}
    for k in (2, 4, 6, 8, 10):
        r[f"@{k} digits"] = round(100 * np.mean([ok(a, b, k) for a, b in zip(p[col], p.gold)]), 1)
    r["avg digits correct"] = round(np.mean([lead(a, b) for a, b in zip(p[col], p.gold)]), 2)
    rows.append(r)
pd.DataFrame(rows).set_index("method")

In [ ]:
# Is fine-tuning significantly better than the baseline? (exact McNemar test on the same 237 rows)
def mcnemar(a, b, k):
    A = np.array([ok(x, g, k) for x, g in zip(p[a], p.gold)]); B = np.array([ok(x, g, k) for x, g in zip(p[b], p.gold)])
    only_b, only_a = int((~A & B).sum()), int((A & ~B).sum())
    return only_b, only_a, binomtest(only_b, only_b + only_a, 0.5).pvalue
for k in (4, 6, 10):
    w, l, pv = mcnemar("tfidf_knn", "qwen_finetuned", k)
    print(f"@{k:<2} digits: fine-tuned right & baseline wrong = {w:3d} | baseline right & fine-tuned wrong = {l:3d} | p = {pv:.3f}")
w, l, pv = mcnemar("qwen_finetuned_no_aug", "qwen_finetuned", 10)
print(f"\nAugmentation ablation @10: fixed {w}, broke {l}, p = {pv:.3f}")

In [ ]:
# Accuracy by part type (exact 10-digit), headings with >= 6 test rows
NAMES = {"8541": "Diodes, transistors, LEDs, crystals", "8542": "Integrated circuits",
         "8536": "Connectors, switches, sockets", "8532": "Capacitors", "8533": "Resistors",
         "8504": "Inductors", "8548": "Ferrite beads"}
p["heading"] = p.gold.str[:4]
t = (p.groupby("heading")
       .apply(lambda g: pd.Series({"rows": len(g),
             "TF-IDF %": round(100 * np.mean([ok(a, b, 10) for a, b in zip(g.tfidf_knn, g.gold)]), 1),
             "fine-tuned %": round(100 * np.mean([ok(a, b, 10) for a, b in zip(g.qwen_finetuned, g.gold)]), 1)}))
       .query("rows >= 6").sort_values("rows", ascending=False))
t["rows"] = t["rows"].astype(int)
t.insert(0, "part type", [NAMES.get(h, "") for h in t.index])
t

In [ ]:
# Where fine-tuning fixes the baseline, and where it still fails
fix = p[[ok(a, g, 10) and not ok(b, g, 10) for a, b, g in zip(p.qwen_finetuned, p.tfidf_knn, p.gold)]]
miss = p[[not ok(a, g, 4) for a, g in zip(p.qwen_finetuned, p.gold)]]
print("Fine-tuned right, baseline wrong (sample):")
display(fix[["text", "gold", "tfidf_knn", "qwen_finetuned"]].head(6))
print("Fine-tuned wrong even at the 4-digit heading (sample):")
display(miss[["text", "gold", "qwen_finetuned"]].head(6))

## Part 3: The live model

Loads the base **Qwen2.5-3B-Instruct** in 4-bit and attaches the **QLoRA adapter** trained on the BOM dataset (29.9 M trainable parameters, 0.96% of the model). The same weights with the adapter switched off give the zero-shot baseline, so you can see exactly what fine-tuning added. Loading takes about 2 minutes.

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel

BASE = "Qwen/Qwen2.5-3B-Instruct"
DTYPE = torch.bfloat16 if torch.cuda.get_device_capability()[0] >= 8 else torch.float16
tok = AutoTokenizer.from_pretrained(BASE)
base = AutoModelForCausalLM.from_pretrained(
    BASE, device_map="auto", torch_dtype=DTYPE,
    quantization_config=BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                                           bnb_4bit_use_double_quant=True, bnb_4bit_compute_dtype=DTYPE))
model = PeftModel.from_pretrained(base, "qwen_hts_lora_adapter")
model.eval()

SYS = ("You are a US customs classification assistant. Given an engineering bill-of-materials "
       "description of a part, reply with only its 10-digit US HTS code in the form XXXX.XX.XXXX.")

@torch.no_grad()
def classify(text, use_adapter=True):
    prompt = tok.apply_chat_template([{"role": "system", "content": SYS}, {"role": "user", "content": text}],
                                     tokenize=False, add_generation_prompt=True)
    enc = tok(prompt, return_tensors="pt", add_special_tokens=False).to(model.device)
    gen = lambda: model.generate(**enc, max_new_tokens=16, do_sample=False, pad_token_id=tok.eos_token_id)
    if use_adapter:
        out = gen()
    else:
        with model.disable_adapter():
            out = gen()
    return tok.decode(out[0, enc["input_ids"].shape[1]:], skip_special_tokens=True).strip()

HEADINGS = {"8532": "capacitor", "8533": "resistor", "8541": "diode / transistor / LED / crystal",
            "8542": "integrated circuit", "8536": "connector / switch / socket", "8504": "inductor",
            "8548": "ferrite bead", "7318": "screw / nut / standoff", "8517": "radio / network module"}
explain = lambda code: HEADINGS.get(re.sub(r"\D", "", code)[:4], "other heading")
print("model ready on", torch.cuda.get_device_name(0))

In [ ]:
# Before vs after fine-tuning, on real test-set descriptions
examples = ["600R L_0402", "100n Bypass 16V/20% C_0402", "390nH", "4.7uf tantalum 3528",
            "3V3REG LDO CMOS HICURR U-DFN3030-8", "FerriteBead L_0805_2012Metric", "SS310 D_SMA"]
print(f"{'input':36} {'no fine-tuning':16} {'fine-tuned':14} part type")
for t in examples:
    z, f = classify(t, use_adapter=False), classify(t)
    print(f"{t:36} {z[:15]:16} {f:14} {explain(f)}")

## Part 4: Try it

Type a BOM-style description (for example `10uF 25V X5R 0603`, `M3 nylon standoff 10mm`, `IC MCU 32BIT 64KB FLASH`). Press Enter on an empty line to stop.

In [ ]:
while True:
    t = input("BOM description> ").strip()
    if not t:
        break
    c = classify(t)
    print(f"  -> {c}   ({explain(c)})")